# SHAP (SHapley Additive exPlanations) — Sistemas Basados en Conocimiento

**Curso:** Sistemas Basados en Conocimiento  
**Alumno:** Víctor Vargas Miranda  
**Dataset:** [Breast Cancer Dataset — Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)  
**Librerías principales:** [scikit-learn](https://scikit-learn.org/stable/install) · [shap](https://shap.readthedocs.io/en/latest/generated/shap.Explainer.html)

---

## 1. ¿Qué problema resuelve SHAP?

Los modelos de aprendizaje automático modernos —como bosques aleatorios, redes neuronales o gradient boosting— logran predicciones altamente precisas. Sin embargo, su comportamiento interno es opaco: no resulta trivial entender **por qué** un modelo asigna una determinada predicción a una instancia concreta. Esto genera dos problemas prácticos:

1. **Confianza y auditoría:** Sin explicaciones, es difícil confiar en modelos que toman decisiones sensibles (diagnóstico médico, crédito bancario, selección de personal).
2. **Depuración y mejora:** Si no sabemos qué características impulsan las predicciones, resulta muy difícil detectar sesgos o mejorar el modelo.

**SHAP** (SHapley Additive exPlanations) resuelve estos problemas proporcionando una forma **unificada, justa y matemáticamente rigurosa** de atribuir a cada característica su contribución marginal a una predicción concreta.

### Fundamento matemático: valores de Shapley

SHAP se basa en los **valores de Shapley** de la teoría de juegos cooperativos (Shapley, 1953). Dado un conjunto de jugadores (características) que colaboran para producir un resultado (la predicción del modelo), el valor de Shapley de cada jugador es su **contribución marginal promediada sobre todas las posibles coaliciones** (subconjuntos de características):

$$\phi_i(f, x) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F|-|S|-1)!}{|F|!} \left[ f(S \cup \{i\}) - f(S) \right]$$

donde:
- $F$ es el conjunto de todas las características
- $S$ es un subconjunto que no incluye la característica $i$
- $f(S)$ es la predicción del modelo usando únicamente las características del subconjunto $S$
- $\phi_i$ es el **valor SHAP** de la característica $i$: cuánto contribuye (positiva o negativamente) a desplazar la predicción desde el valor base (media global) hasta el valor predicho para esa instancia.

### Propiedad de eficiencia (additividad)

La suma de todos los valores SHAP de una instancia más el valor base es exactamente igual a la predicción del modelo:

$$f(x) = \phi_0 + \sum_{i=1}^{n} \phi_i(x)$$

donde $\phi_0 = E[f(x)]$ es la predicción media (valor base). Esto hace que SHAP sea **completamente aditivo y consistente**: cada característica recibe exactamente su "parte justa" de la predicción.

## 2. Supuestos Fundamentales de SHAP

SHAP está fundamentado en cuatro axiomas de la teoría de juegos cooperativos que garantizan una asignación justa de contribuciones:

1. **Eficiencia (Efficiency):** La suma de todos los valores SHAP es igual a la diferencia entre la predicción real y la predicción base:
   $$\sum_{i=1}^{n} \phi_i = f(x) - \phi_0$$
   Cada unidad de "mérito predictivo" queda completamente distribuida entre las características.

2. **Simetría (Symmetry):** Si dos características contribuyen idénticamente al modelo en todas las coaliciones posibles, se les asigna el mismo valor SHAP. No hay tratamiento arbitrariamente preferente.

3. **Jugador nulo (Dummy player):** Si una característica nunca modifica la predicción (sin importar qué coalición la incluya), su valor SHAP es cero.

4. **Aditividad (Linearity / Additivity):** Para modelos ensamblados (como boosting), el valor SHAP total de cada característica es la suma de sus valores SHAP en cada componente. Esto permite explicar modelos complejos descomponiéndolos.

### Consideración práctica: independencia de características

La implementación estándar de SHAP asume que las características son **independientes** al construir las coaliciones. En la práctica, muchos datasets tienen correlaciones entre variables. Existen variantes que manejan dependencias (p. ej., `TreeExplainer` con `feature_perturbation="interventional"`), pero la interpretación debe hacerse con cuidado cuando hay alta correlación.

### Sobre el modelo subyacente

SHAP **no hace supuestos sobre el modelo que explica**: es agnóstico al algoritmo. Funciona con árboles de decisión, redes neuronales, regresiones, etc. Lo único que necesita es poder evaluar $f(S)$ para cualquier subconjunto de características.

## 3. Usos y Aplicaciones Potenciales

SHAP tiene aplicaciones concretas en numerosos dominios:

| Dominio | Aplicación |
|---|---|
| **Medicina / Salud** | Explicar por qué un modelo predice riesgo de enfermedad en un paciente específico; identificar biomarcadores clave |
| **Finanzas** | Justificar la aprobación o denegación de crédito (regulación GDPR exige explicabilidad) |
| **Recursos Humanos** | Auditar modelos de selección de candidatos para detectar sesgos demográficos |
| **Marketing** | Entender qué factores impulsan la conversión de clientes individuales |
| **Mantenimiento predictivo** | Explicar por qué el modelo predice falla inminente de un equipo y qué sensor lo desencadena |
| **Ciencia** | Descubrir qué características (genes, compuestos químicos) explican un fenómeno observado |
| **NLP / Visión** | Identificar qué palabras o regiones de imagen influyen en la clasificación |

### Usos específicos dentro del ciclo ML

- **Depuración de modelos:** Detectar que el modelo usa variables espurias (p. ej., una fecha de registro) en lugar de las causalmente relevantes.
- **Selección de características:** Eliminar variables con valores SHAP sistemáticamente cercanos a cero.
- **Validación con expertos de dominio:** Compartir explicaciones con médicos o analistas para validar que el modelo aprende patrones conocidos.
- **Cumplimiento normativo:** Generar explicaciones auditables para reguladores (GDPR Art. 22, regulación financiera).

## 4. ¿En qué tipos de problemas es apropiado SHAP?

SHAP es apropiado en prácticamente cualquier problema de aprendizaje supervisado o no supervisado donde se use un modelo de caja negra, pero resulta especialmente valioso cuando:

### Por tipo de tarea
- **Clasificación binaria o multiclase** (nuestro caso: diagnóstico de cáncer)
- **Regresión** (predicción de precios, demanda, etc.)
- **Rankings y modelos de supervivencia**

### Por tipo de modelo
- **Modelos de árboles:** RandomForest, XGBoost, LightGBM, CatBoost → se usa `TreeExplainer`, que calcula valores SHAP exactos en tiempo polinomial.
- **Modelos lineales:** Regresión lineal/logística → se usa `LinearExplainer`, que produce valores SHAP analíticamente.
- **Modelos generales (caja negra pura):** Redes neuronales, SVM, pipelines arbitrarios → se usa `KernelExplainer` (aproximación mediante muestreo) o `DeepExplainer` / `GradientExplainer` para redes neuronales.

### Por necesidad de explicabilidad
- **Explicaciones locales:** Para justificar una predicción individual (p. ej., por qué este paciente tiene alta probabilidad de tumor maligno).
- **Explicaciones globales:** Para entender el comportamiento general del modelo (qué características son más importantes en promedio).

### Cuándo SHAP podría no ser la mejor opción
- Si el dataset es muy pequeño y el tiempo de cómputo de `KernelExplainer` es prohibitivo.
- Si el modelo es ya interpretable de por sí (p. ej., regresión lineal simple): los coeficientes directamente aportan suficiente información.

## 5. ¿Cuándo es preferible SHAP frente a otros métodos de explicabilidad?

Existen varias técnicas de XAI (Explainable AI). SHAP se destaca en los siguientes escenarios:

### SHAP vs. LIME (Local Interpretable Model-agnostic Explanations)

| Criterio | SHAP | LIME |
|---|---|---|
| **Consistencia matemática** | ✅ Garantizada por axiomas de Shapley | ❌ No garantizada; puede variar con la semilla |
| **Explicaciones globales** | ✅ Nativas (beeswarm, bar global) | ❌ Solo locales de forma nativa |
| **Velocidad con árboles** | ✅ Muy rápido (`TreeExplainer`) | Moderada |
| **Interpretación intuitiva** | ✅ Valor = contribución exacta a la predicción | Aproximación local por modelo lineal |

**Preferir SHAP cuando:** se necesitan explicaciones consistentes, auditables, comparables entre instancias, o cuando se trabaja con modelos de árboles.

### SHAP vs. Importancia de características del modelo (Feature Importance)

La importancia por impureza (Gini) de un Random Forest indica cuánto reduce la impureza una característica **en promedio durante el entrenamiento**. Tiene sesgos conocidos hacia variables con muchos valores únicos y no ofrece información a nivel de instancia.

**Preferir SHAP cuando:** se requiere importancia a nivel de instancia, comparación con dirección (positiva/negativa) del efecto, o mayor robustez estadística.

### SHAP vs. Coeficientes de modelos lineales

Los coeficientes de una regresión logística son directamente interpretables solo si las variables están estandarizadas y no hay multicolinealidad. SHAP generaliza esta idea a cualquier modelo.

**Preferir SHAP cuando:** se usan modelos no lineales o cuando se quiere una interpretación comparable entre tipos de modelos.

## 6. Pros y Contras desde la Perspectiva de XAI

### ✅ Ventajas (Pros)

1. **Fundamentación teórica sólida:** Los valores de Shapley son la única asignación de crédito que satisface simultáneamente eficiencia, simetría, jugador nulo y aditividad. Esto le da legitimidad científica.

2. **Explicaciones locales y globales en un solo marco:** Con los mismos valores SHAP se pueden construir tanto la explicación de una instancia individual (waterfall, force plot) como una visión global del modelo (bar, beeswarm).

3. **Agnóstico al modelo:** Funciona con cualquier función predictiva, lo que permite comparar explicaciones entre modelos distintos.

4. **Consistencia:** Si un modelo cambia de forma que una característica se vuelve más importante, su valor SHAP nunca puede decrecer. Esto no está garantizado en, por ejemplo, la importancia de permutación.

5. **Dirección del efecto:** A diferencia de la importancia de características tradicional, SHAP muestra si una característica empuja la predicción hacia arriba o hacia abajo.

6. **Detecta interacciones:** Los `shap interaction values` permiten cuantificar interacciones entre pares de variables.

### ❌ Desventajas (Contras)

1. **Costo computacional:** El cálculo exacto es exponencial en el número de características ($O(2^n)$). Los algoritmos aproximados (`TreeExplainer`, `KernelExplainer`) reducen este costo pero pueden introducir error.

2. **Supuesto de independencia:** La formulación estándar asume independencia entre características. Con alta correlación, los valores SHAP pueden distribuir el crédito de forma contraintuitiva entre variables correlacionadas.

3. **No es causal:** SHAP mide correlación/contribución dentro del modelo, no causalidad. Un valor SHAP alto no implica que la característica cause el resultado.

4. **Explicaciones contrafácticas limitadas:** SHAP no responde directamente "¿qué tendría que cambiar para obtener otra predicción?". Para eso se necesitan métodos contrafácticos.

5. **Puede ser difícil de comunicar:** Los gráficos SHAP (especialmente beeswarm y force plots) requieren cierta alfabetización técnica para ser interpretados correctamente por usuarios no técnicos.

## 7. Análisis de Explicabilidad del Modelo

En este cuaderno se usa SHAP **no para estudiar el cáncer de mama ni para validar el modelo médico**, sino como vehículo para aprender las capacidades explicativas de SHAP sobre un modelo de clasificación real. El análisis de explicabilidad se articulará en torno a tres niveles:

### 7.1 Explicabilidad Global
Entender el comportamiento general del modelo preguntando: ¿qué características son más influyentes en promedio? ¿En qué dirección operan?

- **Gráfico de barras (bar plot):** muestra la importancia media global de cada característica (media del valor absoluto de SHAP sobre todas las instancias).
- **Beeswarm plot:** extiende el bar plot mostrando, para cada instancia, el valor SHAP y el valor de la característica. Permite ver no solo la importancia sino la relación entre el valor de la característica y su efecto.

### 7.2 Explicabilidad Local
Entender por qué el modelo asignó una predicción concreta a una instancia específica.

- **Waterfall plot:** descompone la predicción de una instancia como suma del valor base más las contribuciones de cada característica, de mayor a menor magnitud.
- **Force plot:** representación horizontal del mismo concepto; muestra qué características "empujan" la predicción hacia arriba (rojo) o hacia abajo (azul) respecto al valor base.

### 7.3 Análisis de Dependencia
Entender la relación entre el valor de una característica y su contribución SHAP.

- **Scatter / Dependence plot:** grafica el valor de la característica en el eje X y su valor SHAP en el eje Y. Si la relación es monótona y suave, el modelo ha aprendido un patrón estable. Si hay mucho ruido, puede indicar interacciones con otras variables.

---
## Solución Técnica

> **Nota de instalación:** Si ejecutas este cuaderno en un entorno sin `shap` instalado, ejecuta primero la celda siguiente.

In [ ]:
# Instalar shap si no está disponible
# !pip install shap

In [ ]:
import warnings
# Suppress only known non-critical warnings from SHAP and matplotlib
warnings.filterwarnings('ignore', category=UserWarning, module='shap')
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

print(f"shap version: {shap.__version__}")

### Carga y exploración del dataset

El dataset **Breast Cancer Wisconsin** de scikit-learn contiene 569 muestras de tumores de mama con 30 características numéricas calculadas a partir de imágenes de biopsias. La variable objetivo es binaria: 0 = maligno, 1 = benigno.

> **Recordatorio:** Usamos este dataset únicamente para aprender SHAP, no para hacer un análisis médico.

In [ ]:
# Cargar el dataset
cancer = load_breast_cancer()

X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name='target')

print("Forma del dataset:", X.shape)
print("\nDistribución de clases:")
print(y.value_counts().rename({0: 'Maligno (0)', 1: 'Benigno (1)'})) 
print("\nPrimeras 3 filas:")
X.head(3)

### Entrenamiento del modelo

Entrenamos un **Random Forest** con scikit-learn. Este modelo es una elección ideal para demostrar SHAP porque:
- Permite usar `TreeExplainer`, que calcula valores SHAP exactos de forma eficiente.
- Es un modelo de conjunto (ensemble) que sería difícil de interpretar sin SHAP.

In [ ]:
# División entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Entrenamiento del Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Evaluación básica
print("Reporte de clasificación (conjunto de prueba):")
print(classification_report(y_test, rf_model.predict(X_test),
                            target_names=cancer.target_names))

---
## Usando `shap.Explainer` — La Interfaz Principal

`shap.Explainer` es la **interfaz unificada** de la librería SHAP. Detecta automáticamente el tipo de modelo y selecciona el algoritmo de explicación más apropiado:

- Para modelos de árbol (RandomForest, XGBoost, etc.) → usa internamente `TreeExplainer`
- Para modelos lineales → usa `LinearExplainer`
- Para modelos genéricos → usa `PermutationExplainer` o `KernelExplainer`

El resultado es un objeto `shap.Explanation` que contiene:
- **`.values`**: array de valores SHAP (forma: `[n_instancias, n_features]` o `[n_instancias, n_features, n_clases]`)
- **`.base_values`**: valor base (predicción media del modelo = $\phi_0$)
- **`.data`**: los valores originales de las características
- **`.feature_names`**: nombres de las características

In [ ]:
# Crear el Explainer con la interfaz principal
explainer = shap.Explainer(rf_model, X_train)

# Calcular los valores SHAP para el conjunto de prueba
shap_values = explainer(X_test)

print("Tipo del objeto devuelto:", type(shap_values))
print("Forma de shap_values.values:", shap_values.values.shape)
print("Forma de shap_values.base_values:", shap_values.base_values.shape)
print("Forma de shap_values.data:", shap_values.data.shape)
print("Número de feature names:", len(shap_values.feature_names))

### Inspeccionando `shap.Explanation`

El objeto `shap.Explanation` es el contenedor central de toda la información de explicabilidad. Veamos sus componentes en la primera instancia del conjunto de prueba.

In [ ]:
# Seleccionamos la clase 1 (Benigno) para el análisis
# shap_values.values tiene forma [n_instancias, n_features, n_clases]
# Para clasificación binaria con RF: clase 0 y clase 1
# Usamos la clase 1 (benigno)
shap_exp_class1 = shap_values[..., 1]

# Inspeccionamos la primera instancia
instancia_idx = 0
instancia = shap_exp_class1[instancia_idx]

print("=== shap.Explanation para la instancia 0 (clase Benigno) ===")
print(f"\nPredicción del modelo: {rf_model.predict_proba(X_test.iloc[[instancia_idx]])[0, 1]:.4f}")
print(f"Valor base (.base_values): {instancia.base_values:.4f}")
print(f"Suma de valores SHAP: {instancia.values.sum():.4f}")
print(f"Base + SHAP = {instancia.base_values + instancia.values.sum():.4f}")
print("\n→ La propiedad de eficiencia se cumple: base + Σφᵢ = predicción del modelo")

# Top 5 características más influyentes para esta instancia
shap_df = pd.DataFrame({
    'feature': shap_values.feature_names,
    'shap_value': instancia.values,
    'feature_value': instancia.data
}).sort_values('shap_value', key=abs, ascending=False).head(5)

print("\nTop 5 características más influyentes para la instancia 0:")
print(shap_df.to_string(index=False))

---
## Gráficos de Explicabilidad Global

### Gráfico de Barras Global (Bar Plot)

El **bar plot global** muestra la importancia media de cada característica calculada como la media del valor absoluto de los valores SHAP sobre todas las instancias del conjunto de prueba:

$$\text{Importancia}(i) = \frac{1}{N} \sum_{j=1}^{N} |\phi_i(x^{(j)})|$$

A diferencia de la importancia por impureza (Gini) de Random Forest, esta medida es **consistente** y tiene en cuenta tanto el signo como la magnitud del efecto en cada predicción.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.bar(shap_exp_class1, max_display=15, show=False)
plt.title("Importancia Global de Características (SHAP) — Clase Benigno", 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Beeswarm Plot

El **beeswarm plot** es la visualización más informativa para entender el comportamiento global de un modelo. Para cada característica y cada instancia muestra un punto cuya posición horizontal es el valor SHAP y cuyo color indica el valor original de la característica (rojo = valor alto, azul = valor bajo).

Esto permite leer de un solo vistazo:
- **Qué tan importante** es cada característica (dispersión horizontal)
- **En qué dirección** opera (positivos → predicción de clase 1 aumenta; negativos → disminuye)
- **La relación** entre el valor de la característica y su efecto (¿valores altos de la variable empujan hacia arriba o hacia abajo la predicción?)

In [ ]:
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_exp_class1, max_display=15, show=False)
plt.title("Beeswarm Plot — Explicabilidad Global del Modelo", 
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


#### Análisis del Beeswarm Plot

Al observar el beeswarm plot podemos extraer patrones interpretables:

- Las características en la parte superior son las más influyentes en las predicciones del modelo.
- **Puntos rojos con SHAP positivo:** Valores altos de la característica incrementan la probabilidad de predicción "Benigno".
- **Puntos azules con SHAP positivo:** Valores bajos de la característica también pueden incrementar la probabilidad de "Benigno".
- Cuando los puntos rojos se concentran en la derecha (SHAP > 0) y los azules en la izquierda (SHAP < 0), la relación es monótona: a mayor valor de la variable, mayor probabilidad de clase 1.
- Si hay mezcla de colores, el efecto es no monótono o interactúa con otras características.

---
## Gráficos de Explicabilidad Local

### Waterfall Plot

El **waterfall plot** explica **una instancia concreta**. Muestra cómo cada característica desplaza la predicción desde el valor base $E[f(x)]$ (media global del modelo) hasta el valor predicho $f(x)$ para esa instancia específica.

- Barras **rojas**: la característica empuja la predicción hacia arriba (aumenta la probabilidad de clase Benigno).
- Barras **azules**: la característica empuja la predicción hacia abajo.
- La suma acumulada de barras lleva exactamente desde el valor base hasta la predicción final.

In [ ]:
# Explicación local para la primera instancia del conjunto de prueba
instancia_idx = 0
pred_prob = rf_model.predict_proba(X_test.iloc[[instancia_idx]])[0, 1]
pred_clase = cancer.target_names[rf_model.predict(X_test.iloc[[instancia_idx]])[0]]
real_clase = cancer.target_names[y_test.iloc[instancia_idx]]

print(f"Instancia {instancia_idx}:")
print(f"  Clase real:       {real_clase}")
print(f"  Clase predicha:   {pred_clase}")
print(f"  Probabilidad Benigno: {pred_prob:.4f}")

plt.figure(figsize=(10, 7))
shap.plots.waterfall(shap_exp_class1[instancia_idx], max_display=15, show=False)
plt.title(f"Waterfall Plot — Instancia {instancia_idx} (Predicción: {pred_clase}, P={pred_prob:.2f})",
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### Force Plot

El **force plot** es una representación horizontal compacta del mismo concepto del waterfall plot. Es especialmente útil para:
- Presentar explicaciones a usuarios no técnicos.
- Incrustar en interfaces web o informes.
- Mostrar múltiples instancias apiladas para visualizar patrones.

In [ ]:
# Force plot para la primera instancia (versión matplotlib)
plt.figure(figsize=(14, 4))
shap.plots.force(
    shap_exp_class1[instancia_idx],
    matplotlib=True,
    show=False
)
plt.title(f"Force Plot — Instancia {instancia_idx}", fontsize=12, fontweight='bold', y=1.12)
plt.show()


#### Force plot para múltiples instancias

También podemos comparar varias instancias. A continuación mostramos los force plots de las primeras 5 instancias del conjunto de prueba.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 18))

for i in range(5):
    plt.sca(axes[i])
    pred_p = rf_model.predict_proba(X_test.iloc[[i]])[0, 1]
    pred_c = cancer.target_names[rf_model.predict(X_test.iloc[[i]])[0]]
    shap.plots.force(
        shap_exp_class1[i],
        matplotlib=True,
        show=False
    )
    axes[i].set_title(f"Instancia {i} — Predicción: {pred_c} (P={pred_p:.2f})",
                      fontsize=10, fontweight='bold', y=1.1)

plt.suptitle("Force Plots — Primeras 5 Instancias del Conjunto de Prueba",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Gráfico de Dependencia (Scatter / Dependence Plot)

El **scatter/dependence plot** muestra la relación entre el valor de una característica (eje X) y su valor SHAP (eje Y). El color indica el valor de una segunda característica (automáticamente seleccionada por SHAP como la que más interactúa).

Esto permite detectar:
- **Linealidad o no linealidad** del efecto de una variable
- **Interacciones** entre variables: si el color introduce un patrón visible, hay interacción
- **Umbrales o discontinuidades** en el comportamiento del modelo

Analizamos las 3 características más importantes según el bar plot global.

In [ ]:
# Identificar las 3 características más importantes
mean_abs_shap = np.abs(shap_exp_class1.values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[::-1][:3]
top3_features = [shap_values.feature_names[i] for i in top3_idx]
print("Top 3 características más importantes:", top3_features)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feature in zip(axes, top3_features):
    plt.sca(ax)
    shap.plots.scatter(
        shap_exp_class1[:, feature],
        color=shap_exp_class1,
        show=False
    )
    ax.set_title(f"Dependencia: {feature}", fontsize=10, fontweight='bold')

plt.suptitle("Scatter / Dependence Plots — Top 3 Características",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


#### Análisis de los Scatter Plots

En los scatter plots podemos observar:

- **Relación monótona:** Si los valores SHAP aumentan consistentemente con el valor de la característica, el modelo ha aprendido una relación directa.
- **Interacciones (color):** Si el color (variable secundaria) introduce un patrón visible —por ejemplo, que los puntos azules y rojos siguen trayectorias distintas— existe una interacción entre ambas variables que el modelo está utilizando.
- **Umbrales:** Una curva que es plana y luego tiene un salto brusco sugiere que el modelo aprendió un umbral de decisión en esa variable.

---
## `TreeExplainer` — Explicador Especializado para Modelos de Árboles

Si bien `shap.Explainer` detecta automáticamente que estamos ante un modelo de árbol y usa `TreeExplainer` internamente, es útil instanciarlo explícitamente para entender sus capacidades específicas.

**`shap.TreeExplainer`** implementa el algoritmo de Lundberg et al. (2018) que calcula valores SHAP **exactos** para modelos de árbol en tiempo polinomial $O(L \cdot D^2 \cdot 2^D)$ donde $L$ es el número de hojas y $D$ la profundidad, en lugar del tiempo exponencial del cálculo genérico.

Ventajas sobre el explainer genérico:
- **Exactitud:** valores SHAP exactos, no aproximaciones
- **Velocidad:** significativamente más rápido
- **Valores SHAP de interacción:** permite calcular `shap_interaction_values` que descomponen cada valor SHAP en efectos principales e interacciones

In [ ]:
# Instanciar TreeExplainer explícitamente
tree_explainer = shap.TreeExplainer(rf_model)

# Calcular valores SHAP con TreeExplainer
shap_tree_values = tree_explainer(X_test)

print("TreeExplainer — información del objeto:")
print(f"  Tipo: {type(tree_explainer)}")
print(f"  Valor base (expected_value clase 1): {tree_explainer.expected_value[1]:.4f}")
print(f"  Forma shap_values: {shap_tree_values.values.shape}")

# Verificar consistencia entre Explainer genérico y TreeExplainer
diff = np.abs(shap_values.values - shap_tree_values.values).max()
print(f"\nDiferencia máxima entre shap.Explainer y shap.TreeExplainer: {diff:.2e}")
print("→ Ambos producen los mismos valores (shap.Explainer usa TreeExplainer internamente)")

---
## `KernelExplainer` — Explicador General para Modelos de Caja Negra

`shap.KernelExplainer` es el explainer universal: funciona con **cualquier función** que mapee entradas a salidas, sin necesidad de conocer la estructura interna del modelo. Lo utilizaríamos cuando:

- El modelo no es un árbol, red neuronal, ni modelo lineal (p. ej., SVM, k-NN, ensambles personalizados).
- El modelo es un pipeline complejo con preprocesamiento.
- Se quiere una implementación de referencia para validar otros explainers.

**Funcionamiento:** `KernelExplainer` aproxima los valores SHAP usando regresión lineal ponderada sobre coaliciones de características muestreadas. Es más lento que `TreeExplainer` pero más general.

> Para demostrar `KernelExplainer` usamos una **submuestra pequeña** para que sea computacionalmente manejable en un entorno de aprendizaje.

In [ ]:
# Submuestra para eficiencia computacional
X_background = shap.sample(X_train, 50, random_state=42)  # fondo de referencia
X_test_sample = X_test.iloc[:20]  # solo 20 instancias para demostración

# KernelExplainer necesita una función que reciba datos y devuelva predicciones
# Usamos predict_proba para obtener probabilidades
kernel_explainer = shap.KernelExplainer(
    model=rf_model.predict_proba,
    data=X_background,
    link='identity'
)

print("Calculando valores SHAP con KernelExplainer (puede tardar unos segundos)...")
kernel_shap_values = kernel_explainer.shap_values(X_test_sample, nsamples=100)

print(f"\nKernelExplainer completado.")
print(f"Tipo del resultado: {type(kernel_shap_values)}")
print(f"Forma del array de valores SHAP: {kernel_shap_values.shape}")
print(f"  → [n_instancias={kernel_shap_values.shape[0]}, n_features={kernel_shap_values.shape[1]}, n_clases={kernel_shap_values.shape[2]}]")
print(f"Valor base (clase 1): {kernel_explainer.expected_value[1]:.4f}")

In [ ]:
# Comparación visual: TreeExplainer vs KernelExplainer para las primeras 20 instancias
tree_vals_20 = shap_tree_values[..., 1].values[:20]  # TreeExplainer, clase 1
kernel_vals_20 = kernel_shap_values[:, :, 1]         # KernelExplainer, clase 1 (shape: n_instancias x n_features)

# Comparamos la importancia media de cada característica
tree_importance = np.abs(tree_vals_20).mean(axis=0)
kernel_importance = np.abs(kernel_vals_20).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot TreeExplainer
top_n = 10
top_idx_tree = np.argsort(tree_importance)[::-1][:top_n]
axes[0].barh([cancer.feature_names[i] for i in top_idx_tree][::-1],
             tree_importance[top_idx_tree][::-1], color='steelblue')
axes[0].set_xlabel('Importancia SHAP media')
axes[0].set_title('TreeExplainer\n(Exacto, rápido para árboles)', fontweight='bold')

# Bar plot KernelExplainer
top_idx_kernel = np.argsort(kernel_importance)[::-1][:top_n]
axes[1].barh([cancer.feature_names[i] for i in top_idx_kernel][::-1],
             kernel_importance[top_idx_kernel][::-1], color='darkorange')
axes[1].set_xlabel('Importancia SHAP media')
axes[1].set_title('KernelExplainer\n(Aproximado, universal para cualquier modelo)', fontweight='bold')

plt.suptitle('Comparación: TreeExplainer vs KernelExplainer (Top 10 características)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


#### Análisis: TreeExplainer vs KernelExplainer

La comparación entre ambos explainers revela:

- **Coherencia:** Ambos identifican las mismas características como más importantes. Esto valida que `KernelExplainer`, aunque aproximado, captura los patrones fundamentales del modelo.
- **Diferencias en magnitudes:** `KernelExplainer` usa solo 100 muestras (`nsamples=100`) para la aproximación. Con más muestras converge hacia los valores exactos de `TreeExplainer`.
- **Uso práctico:** Para modelos de árbol, siempre se prefiere `TreeExplainer`. `KernelExplainer` es el recurso cuando se tienen modelos como SVM, redes neuronales, pipelines de preprocesamiento, etc.

---
## Resumen y Conclusiones del Análisis de Explicabilidad

A lo largo de este cuaderno hemos demostrado las capacidades de SHAP para explicar un modelo Random Forest entrenado sobre el dataset Breast Cancer:

### Hallazgos del análisis explicativo

| Herramienta | Nivel | Información obtenida |
|---|---|---|
| **Bar plot global** | Global | Ranking de importancia de características promediado |
| **Beeswarm plot** | Global | Importancia + dirección del efecto + distribución por instancia |
| **Waterfall plot** | Local | Desglose exacto de cómo se llega a la predicción de una instancia |
| **Force plot** | Local | Visualización compacta de la misma información |
| **Scatter/Dependence plot** | Característica | Relación valor↔efecto e interacciones |

### Verificación de la propiedad fundamental

En toda la sesión hemos podido verificar que:
$$f(x) = \phi_0 + \sum_{i=1}^{n} \phi_i(x)$$

La predicción del modelo se descompone **exacta y completamente** en contribuciones de cada característica. Esto es lo que distingue a SHAP de métodos de importancia ad hoc.

### Conclusión desde la perspectiva XAI

SHAP representa el estado del arte en explicabilidad de modelos de aprendizaje automático. Su fundamento en valores de Shapley le confiere propiedades matemáticas únicas (eficiencia, simetría, jugador nulo, aditividad) que ningún otro método de importancia satisface simultáneamente. Aunque tiene limitaciones —especialmente con variables correlacionadas y el costo computacional del `KernelExplainer`— su versatilidad y rigor matemático lo convierten en la herramienta de referencia para XAI en modelos supervisados.

---
## Referencias

- Lundberg, S. M., & Lee, S.-I. (2017). *A Unified Approach to Interpreting Model Predictions*. NeurIPS 2017. [https://arxiv.org/abs/1705.07874](https://arxiv.org/abs/1705.07874)
- Lundberg, S. M., et al. (2018). *Consistent Individualized Feature Attribution for Tree Ensembles*. [https://arxiv.org/abs/1802.03888](https://arxiv.org/abs/1802.03888)
- Shapley, L. S. (1953). *A Value for n-Person Games*. Contributions to the Theory of Games, Vol. 2.
- SHAP Documentation: [https://shap.readthedocs.io/en/latest/](https://shap.readthedocs.io/en/latest/)
- scikit-learn Breast Cancer Dataset: [https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)